# Contrastive Probe Inference

Runs the spatial-grounding probe as a factorial of control conditions over two
scene sources, logging both the executable action and a continuous readout.

Three changes separate this from the earlier probe, each addressing a defect
that made the previous results uninterpretable.

**Continuous readout.** OpenVLA emits one token per action dimension and decodes
it to a bin centre, so the executable action is quantised. The lateral bin is
roughly 1e-3 wide while observed lateral predictions sit between 1e-4 and 4e-3
of zero, which put the earlier median paired difference at exactly zero in every
stratum with several pairs bit-identical between the two instructions. Alongside
the argmax action, each prediction now records the expected bin centre under the
model's own distribution over action tokens, which resolves differences smaller
than one bin. The `a*` columns keep their previous meaning, so earlier logs stay
comparable.

**Control conditions.** The original contrast alone cannot support a conclusion.
A model mapping the token `left` to a leftward action without consulting the
image reproduces the expected sign flip exactly, and a model whose lateral output
ignores the image produces no difference for reasons unrelated to language. Each
condition varies one factor: mirroring reverses the lateral axis with the
instruction held fixed, pairing an instruction with another scene removes its
referent while leaving the language intact, and removing the spatial term gives a
within-scene reference.

**Two scene sources.** `constructed` scenes carry the primary analysis, because
they hold two instances of the target noun and their geometry is recorded rather
than inferred. `bridge` scenes are retained as the secondary external-validity
analysis.

Output is `probe_predictions_v4.csv`. New columns relative to v3: `scene_source`,
`condition`, `configuration`, `axis_index`, `image_transform`, `expected_sign`,
and the continuous readout columns `c0` to `c6` with their diagnostics.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/openvla_cache/hf'
CACHE_DIR = '/content/drive/MyDrive/openvla_cache/bridge_multiobj'
CONSTRUCTED_DIR = '/content/drive/MyDrive/openvla_cache/constructed'
# v2: category and feasible_both, initial frame only.
# v3: adds the frame column ('initial' | 'grasp').
# v4: adds scene_source, condition, configuration, axis_index, and the
#     continuous readout (c0..c6). v3 rows migrate in as baseline predictions
#     rather than being re-run on GPU (see the migration cell below).
PROBE_CSV_V3 = '/content/drive/MyDrive/openvla_cache/probe_predictions_v3.csv'
PROBE_CSV = '/content/drive/MyDrive/openvla_cache/probe_predictions_v4.csv'
print('cache       ->', CACHE_DIR)
print('constructed ->', CONSTRUCTED_DIR)
print('log         ->', PROBE_CSV)

## 2. Import the code

Clones the project code from GitHub into the runtime and imports the loader,
inference, control, and logging functions from there, so the code always matches
the pushed commit.

In [ ]:
import sys, os, importlib, subprocess

REPO_URL = 'https://github.com/LewisTL/ECS8056.git'
BRANCH = 'master'
REPO_DIR = '/content/ECS8056'

def sync_repo():
    """Clone or hard-refresh the repository so it matches origin/BRANCH."""
    token = os.environ.get('GITHUB_TOKEN', '')
    url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', url],
                       check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--quiet', '--depth', '1',
                        'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', '--quiet',
                        f'origin/{BRANCH}'], check=True)
    else:
        subprocess.run(['git', 'clone', '--quiet', '--depth', '1', '--branch',
                        BRANCH, url, REPO_DIR], check=True)
    return subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'],
                          capture_output=True, text=True).stdout.strip()


commit = sync_repo()
module_dir = REPO_DIR
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)
for _m in ('action_bins', 'prediction_log', 'model', 'data', 'controls', 'compose_scenes', 'export_pairs', 'detect_duplicates', 'analysis',):
    sys.modules.pop(_m, None)
importlib.invalidate_caches()

from model import (load_openvla, predict_action, predict_action_dist,
                   describe_action_space, verify_readout, run_metadata,
                   append_prediction_log)
from data import load_manifest, make_pair, term_axis, term_axis_index
from controls import (plan_stimuli, build_scene_swap, apply_image_transform,
                      strip_spatial_term, DEFAULT_CONDITIONS)
from compose_scenes import load_constructed_manifest, IMAGE_X_TO_LATERAL_SIGN
import analysis
print(f'imported project modules from {module_dir} @ {commit}')

## 3. Load OpenVLA-7B

In [ ]:
processor, vla, compute_dtype = load_openvla(quantize_4bit=True, precision='bf16')
meta = run_metadata(compute_dtype)
print(meta)

## 4. Action space and readout verification

The continuous readout reimplements the decoding OpenVLA performs inside
`predict_action`, against constants that live in its remote modelling code and
can move between revisions. Those constants are printed here rather than
assumed, and the reimplementation is then required to reproduce the executable
action exactly on real inputs before any continuous value is used.

`bin_width` is the resolution floor of the argmax readout on each dimension: two
predictions closer together than this cannot differ in the executable action, no
matter how differently the model treats them. It is the number the earlier null
results should have been read against, and it sets the equivalence bound in the
analysis notebook.

In [ ]:
space = describe_action_space(vla)
for key, value in space.items():
    if isinstance(value, list):
        print(f'{key:20} ' + ' '.join(f'{v:+.5f}' if isinstance(v, float) else str(v)
                                      for v in value))
    else:
        print(f'{key:20} {value}')
print()
print(f"lateral (dx) bin width: {space['bin_width'][0]:.6f}")

### Correctness gate

Fifty real scenes are run through both paths. Any disagreement raises, because a
continuous value derived from misidentified constants would be worse than no
continuous value at all: it would look plausible and be wrong.

In [ ]:
from PIL import Image

rows = load_manifest(CACHE_DIR)
gate_samples = []
for row in rows:
    made = make_pair(row.get('instruction', ''))
    if made is None:
        continue
    gate_samples.append((Image.open(os.path.join(CACHE_DIR, row['image_path'])),
                         row['instruction']))
    if len(gate_samples) >= 50:
        break

gate = verify_readout(processor, vla, gate_samples)
assert gate['matched'] == gate['checked'], gate
print('continuous readout verified against predict_action on '
      f"{gate['checked']} scenes")

## 5. Assemble the probe set

Both scene sources are expanded into the same record shape, so the probe loop
and the log schema do not depend on where a scene came from.

For `bridge` scenes the axis comes from the spatial term, and there is no
recorded geometry, so no directional expectation is attached. For `constructed`
scenes the axis is lateral by construction and `expected_sign` is taken from the
manifest geometry, converted from image coordinates by
`IMAGE_X_TO_LATERAL_SIGN`. That constant is the one unresolved link between image
position and action sign; it is applied here in a single place, and the mirror
control establishes it empirically in the analysis notebook.

In [ ]:
probe_set = []

# --- Bridge scenes: the secondary, external-validity source -----------------
for row in rows:
    made = make_pair(row['instruction'])
    if made is None:
        continue
    term, swapped = made
    axis_index = term_axis_index(term)
    if axis_index is None:
        # Scene-dependent relations ("closer to the plate") have no axis that can
        # be fixed before seeing the scene, so they carry no expectation.
        continue
    # category_manual overrides the heuristic category when a scene has been
    # manually reviewed; downstream code follows the same fallback.
    category = row.get('category_manual') or row.get('category', 'other')
    probe_set.append({
        'scene_source': 'bridge',
        'scene_id': f"b{int(row['episode_index']):06d}",
        'pair_id': f"ep{int(row['episode_index']):06d}_{term}",
        'spatial_term': term,
        'axis': term_axis(term),
        'axis_index': axis_index,
        'configuration': '',
        'expected_sign': 0,
        'target_sign_a': 0,
        'target_sign_b': 0,
        'category': category,
        'feasible_both': row.get('feasible_both', 'unreviewed'),
        'duplicate_target': row.get('duplicate_target', 'unreviewed'),
        'image_path': os.path.join(CACHE_DIR, row['image_path']),
        'instr_a': row['instruction'],
        'instr_b': swapped,
    })

# --- Constructed scenes: the primary source ---------------------------------
constructed = []
if os.path.isfile(os.path.join(CONSTRUCTED_DIR, 'constructed_manifest.csv')):
    constructed = load_constructed_manifest(CONSTRUCTED_DIR)
for scene in constructed:
    probe_set.append({
        'scene_source': 'constructed',
        'scene_id': scene['construct_id'],
        'pair_id': scene['construct_id'],
        'spatial_term': scene['spatial_term'],
        'axis': 'lateral',
        'axis_index': 0,
        'configuration': scene['configuration'],
        'expected_sign': int(scene['expected_sign_image']) * IMAGE_X_TO_LATERAL_SIGN,
        # The side each instruction's own target sits on. Scoring each
        # instruction against its own target, rather than scoring the pair
        # against their relative order, is what separates scene grounding from
        # a fixed word-to-direction mapping on the same-side configurations.
        'target_sign_a': int(scene['target_sign_a_image']) * IMAGE_X_TO_LATERAL_SIGN,
        'target_sign_b': int(scene['target_sign_b_image']) * IMAGE_X_TO_LATERAL_SIGN,
        'category': 'constructed',
        'feasible_both': 'yes',
        'duplicate_target': 'yes',
        'image_path': os.path.join(CONSTRUCTED_DIR, scene['image_path']),
        'instr_a': scene['instr_a'],
        'instr_b': scene['instr_b'],
    })

from collections import Counter
print(f'{len(probe_set)} scenes: ' + str(dict(Counter(
    p['scene_source'] for p in probe_set))))
print('bridge axes:', dict(Counter(p['axis'] for p in probe_set
                                   if p['scene_source'] == 'bridge')))
print('constructed configurations:', dict(Counter(
    p['configuration'] for p in probe_set if p['scene_source'] == 'constructed')))
if not constructed:
    print('\nNo constructed manifest found; run Notebook 02c first. The bridge '
          'source alone supports the instrument check and the secondary analysis.')

## 6. Expand the condition factorial

Every scene is expanded into the predictions its applicable conditions require.
A condition is skipped, not approximated, when its precondition fails: the mirror
conditions need a lateral term because a horizontal flip leaves depth and
vertical relations unchanged, and the term-stripped conditions need a removal
that leaves a well-formed instruction.

Refusals are counted and reported here. A truncated prompt would change the
prediction for reasons unrelated to the spatial term, so producing one would
quietly corrupt the within-scene reference; skipping instead means the neutral
conditions cover a subset of scenes, which the analysis accounts for.

The swapped-scene assignment is a derangement, so no scene is ever paired with
its own image and the control cannot silently degrade into the baseline. It is
built separately per scene source, keeping the replacement image in the same
visual distribution as the original.

In [ ]:
swap_map = {}
for source in ('bridge', 'constructed'):
    ids = [p['scene_id'] for p in probe_set if p['scene_source'] == source]
    if len(ids) >= 2:
        swap_map.update(build_scene_swap(ids, seed=0))

image_lookup = {p['scene_id']: p['image_path'] for p in probe_set}

work = []
refused_strip = 0
for p in probe_set:
    stimuli = plan_stimuli(p, swap_map=swap_map, conditions=DEFAULT_CONDITIONS)
    if strip_spatial_term(p['instr_a'], p['spatial_term']) is None:
        refused_strip += 1
    for stim in stimuli:
        work.append((p, stim))

print(f'{len(work)} predictions across {len(probe_set)} scenes')
print('per condition:', dict(Counter(s.condition for _, s in work)))
print('per scene source:', dict(Counter(p['scene_source'] for p, _ in work)))
print(f'\nterm removal refused on {refused_strip}/{len(probe_set)} scenes '
      f'({refused_strip / max(len(probe_set), 1):.1%}); those scenes contribute '
      'no neutral reference')

for p, stim in work[:8]:
    print(f"  [{p['scene_id']}] {stim.condition:16} {stim.role} "
          f"{stim.image_transform:14} :: {stim.instruction}")

## 7. Migrate the v3 log

Existing v3 rows are copied forward as `baseline` predictions from the `bridge`
source, matching the v2 to v3 precedent, so already-collected predictions are
not re-run on GPU. The continuous columns are left empty for those rows: they
were produced by the argmax path, and back-filling them would require the
distributions, which were never recorded.

The analysis reads the continuous columns where they are populated and reports
coverage, so migrated rows contribute to the argmax comparisons and are excluded
from the continuous ones rather than silently mixed in.

The migrated file is written with the complete v4 header, not with only the
columns the v3 rows happened to have. A log whose header is narrower than the rows
later appended to it is not a CSV any reader can parse, and the failure appears
far from its cause: every append succeeds, and the file breaks only when the
analysis first tries to read it. `PROBE_LOG_FIELDS` is the single declaration of
that header, and the sweep below writes the same set.

In [ ]:
import pandas as pd
from prediction_log import canonical_log_fields, inspect_log, repair_log

# The `**extra` keys the sweep passes, in order. Declared once and used for the
# migrated header, the sweep, and the repair, so the three cannot disagree.
PROBE_EXTRA_FIELDS = [
    'scene_id', 'pair_id', 'role', 'frame', 'scene_source', 'condition',
    'image_transform', 'image_scene_id', 'configuration', 'expected_sign',
    'target_sign_a', 'target_sign_b', 'spatial_term', 'axis', 'axis_index',
    'category', 'feasible_both', 'duplicate_target', 'sample_idx',
]
PROBE_LOG_FIELDS = canonical_log_fields(PROBE_EXTRA_FIELDS)
print(f'v4 schema: {len(PROBE_LOG_FIELDS)} columns')

if os.path.exists(PROBE_CSV):
    print(f'{PROBE_CSV} already exists; skipping migration')
elif os.path.exists(PROBE_CSV_V3):
    prev = pd.read_csv(PROBE_CSV_V3)
    after_role = list(prev.columns).index('role') + 1
    prev.insert(after_role, 'condition', 'baseline')
    prev.insert(after_role, 'scene_source', 'bridge')
    prev['configuration'] = ''
    prev['image_transform'] = 'original'
    prev['expected_sign'] = 0
    prev['target_sign_a'] = 0
    prev['target_sign_b'] = 0
    # Axis recovered from the spatial term, correcting the earlier analysis
    # which read the lateral component for every pair regardless of its axis.
    prev['axis_index'] = prev['spatial_term'].map(term_axis_index)
    prev = prev[prev['axis_index'].notna()].copy()
    prev['axis_index'] = prev['axis_index'].astype(int)
    prev['scene_id'] = prev['scene_id'].apply(lambda s: f'b{int(s):06d}')
    # Reindex onto the full v4 header so the appended rows fit the file.
    prev = prev.reindex(columns=PROBE_LOG_FIELDS)
    prev.to_csv(PROBE_CSV, index=False)
    print(f'migrated {len(prev)} rows from {PROBE_CSV_V3} -> {PROBE_CSV} '
          "(scene_source='bridge', condition='baseline')")
else:
    print('no earlier log to migrate; starting fresh')

## 7b. Check the log is readable

A log written before the header was declared in full can hold rows wider than its
own header, which no CSV reader will parse: the error names a line number and
nothing else. This reports the field counts actually present, and rebuilds the file
when they differ.

The repair reads each row under the schema matching its width and writes them all
out under the union of columns, so no value moves to a different column. Rows whose
width matches no known schema are dropped and counted rather than guessed at, since
values placed under the wrong names would corrupt the analysis silently.

Nothing is re-run on GPU. This is a file-format repair, and it is a no-op on a log
that is already consistent.

In [ ]:
if os.path.exists(PROBE_CSV):
    report = inspect_log(PROBE_CSV)
    print(f"header: {report['header_fields']} columns")
    for width, entry in report['widths'].items():
        print(f"  rows with {width:3} fields: {entry['rows']:6}  "
              f"(first at line {entry['first_line']})")

    if report['consistent']:
        print('\nconsistent; no repair needed')
    else:
        print('\nmixed widths found; repairing')
        repair_log(PROBE_CSV, [PROBE_LOG_FIELDS])
        assert inspect_log(PROBE_CSV)['consistent'], 'repair did not converge'
        print('now readable:', len(pd.read_csv(PROBE_CSV)), 'rows')

## 8. Run the probe

One deterministic prediction per work item, each logged with its condition, role,
scene source, and the axis its comparison will be read on.

Restart-safe: the resume key is
`(scene_source, pair_id, frame, condition, role)`, which extends the v3 key with
the two new factors. Role `n` covers the single-prediction term-stripped
conditions, which have no opposite.

`sample_idx` is fixed at 0 under the deterministic decoding strategy; the column
exists so the schema does not change if repeated sampling is added later.

In [ ]:
import csv
from PIL import Image

FRAME = 'initial'

done = set()
if os.path.exists(PROBE_CSV):
    with open(PROBE_CSV, newline='') as f:
        for r in csv.DictReader(f):
            done.add((r.get('scene_source', 'bridge'), r['pair_id'],
                      r.get('frame') or 'initial',
                      r.get('condition', 'baseline'), r['role']))
    print(f'resuming: {len(done)} predictions already logged')

image_cache = {}
def load_image(scene_id):
    if scene_id not in image_cache:
        image_cache[scene_id] = Image.open(image_lookup[scene_id]).convert('RGB')
    return image_cache[scene_id]

run = 0
for i, (p, stim) in enumerate(work):
    key = (p['scene_source'], p['pair_id'], FRAME, stim.condition, stim.role)
    if key in done:
        continue
    image = apply_image_transform(stim.image_transform,
                                  load_image(stim.image_scene_id))
    readout = predict_action_dist(processor, vla, image, stim.instruction,
                                  compute_dtype)
    # Keyed by PROBE_EXTRA_FIELDS so the row cannot carry a column the declared
    # header lacks, and asserted rather than trusted.
    extra = {
        'scene_id': p['scene_id'],
        'pair_id': p['pair_id'],
        'role': stim.role,
        'frame': FRAME,
        'scene_source': p['scene_source'],
        'condition': stim.condition,
        'image_transform': stim.image_transform,
        'image_scene_id': stim.image_scene_id,
        'configuration': p['configuration'],
        'expected_sign': p['expected_sign'],
        'target_sign_a': p['target_sign_a'],
        'target_sign_b': p['target_sign_b'],
        'spatial_term': p['spatial_term'],
        'axis': p['axis'],
        'axis_index': p['axis_index'],
        'category': p['category'],
        'feasible_both': p['feasible_both'],
        'duplicate_target': p['duplicate_target'],
        'sample_idx': 0,
    }
    assert list(extra) == PROBE_EXTRA_FIELDS, 'log fields drifted from the schema'
    append_prediction_log(PROBE_CSV, readout.action, stim.instruction, meta,
                          readout=readout, **extra)
    done.add(key)
    run += 1
    if run % 100 == 0:
        print(f'{run} predictions run ({i + 1}/{len(work)} work items seen)')

print(f'probe complete: {run} new predictions -> {PROBE_CSV}')

## 9. Determinism check

Twenty stimuli already in the log are re-predicted. Decoding is greedy with fixed
seeds, so the repeats must agree exactly. Establishing that here means any
nonzero difference in the analysis is attributable to the manipulation rather
than to run-to-run variation, which is what lets differences of the size the
continuous readout resolves be taken seriously at all.

In [ ]:
import numpy as np

repeats = []
for p, stim in work[:20]:
    image = apply_image_transform(stim.image_transform,
                                  load_image(stim.image_scene_id))
    again = predict_action_dist(processor, vla, image, stim.instruction,
                                compute_dtype)
    repeats.append((p['scene_id'], stim.condition, stim.role,
                    again.action, again.expected))

log = pd.read_csv(PROBE_CSV)
worst_action, worst_cont, compared = 0.0, 0.0, 0
for scene_id, condition, role, action, expected in repeats:
    match = log[(log['scene_id'] == scene_id) & (log['condition'] == condition)
                & (log['role'] == role)]
    if match.empty:
        continue
    logged_a = match[[f'a{i}' for i in range(7)]].iloc[0].to_numpy(dtype=float)
    logged_c = match[[f'c{i}' for i in range(7)]].iloc[0].to_numpy(dtype=float)
    worst_action = max(worst_action, float(np.max(np.abs(logged_a - action))))
    if np.isfinite(logged_c).all():
        worst_cont = max(worst_cont, float(np.max(np.abs(logged_c - expected))))
    compared += 1

print(f'{compared} stimuli re-predicted')
print(f'largest argmax difference:     {worst_action:.3e}')
print(f'largest continuous difference: {worst_cont:.3e}')
assert worst_action == 0.0 and worst_cont == 0.0, (
    'repeated identical inputs disagreed; decoding is not deterministic and no '
    'difference measured downstream can be attributed to the manipulation')
print('deterministic')

## 10. Instrument check

The gate that decides whether the language measurement can mean anything.

With the instruction held fixed, mirroring the image reverses the lateral axis of
the scene. A model that reads lateral position at all must change the sign of its
lateral output. If it does not, the visual channel is not live on this axis, and
a null on the language comparisons carries no information about spatial language:
it would follow from the model ignoring the image entirely. The term-stripped
variant is the cleaner form, since it isolates object grounding from any
influence of the spatial word.

This check is not itself evidence of spatial language grounding. It establishes
the necessary condition that makes the rest of the analysis interpretable, and it
identifies the lateral axis empirically, which the earlier ground-truth pilot
failed to do.

In [ ]:
log = pd.read_csv(PROBE_CSV)
lateral = log[(log['scene_source'] == 'bridge') & (log['axis_index'] == 0)
              & log['c0'].notna()]
check = analysis.mirror_check(lateral)

for label in ('neutral', 'term'):
    result = check.get(label, {})
    if not result.get('n'):
        print(f'{label}: no paired mirror predictions yet')
        continue
    print(f"[{label}] n={result['n']}")
    print(f"  sign flips under mirroring : {result['flip_rate']:.1%}")
    print(f"  identical to original      : {result['identical_rate']:.1%}")
    print(f"  mean |lateral| original    : {result['mean_abs_original']:.5f}")
    print(f"  mean |change|              : {result['mean_abs_change']:.5f}")
    print(f"  antisymmetry (0 if exact reversal): "
          f"median={result['antisymmetry']['median']:+.5f} "
          f"p={result['antisymmetry']['p_value']:.3g}")
    print(f"  invariance   (0 if ignored)      : "
          f"median={result['invariance']['median']:+.5f} "
          f"p={result['invariance']['p_value']:.3g}")

print('\nRead: a high flip rate with an antisymmetry median near zero means the '
      'lateral channel tracks the scene, and the language comparisons are '
      'interpretable. A near-zero flip rate with an invariance median near zero '
      'means the output ignores the image, and no language conclusion can be '
      'drawn from this axis.')